# **From Fully Connected Layers to Convolutions**
:label:`sec_why-conv`

- The models we have discussed so far remain **appropriate for tabular data**:
  - Tabular data consists of:
    - **Rows** corresponding to examples.
    - **Columns** corresponding to features.
  - In this setting:
    - We may anticipate **interactions among features**.
    - But we **do not assume any prior structure** in how features interact.

- In some cases, we truly **lack domain knowledge** to design advanced architectures:
  - An **MLP may be the best option** under such circumstances.
  - However, for **high-dimensional perceptual data**, such unstructured networks become **unwieldy**.

- **Example: Distinguishing cats from dogs**:
  - Suppose we use **one-megapixel images** as input.
  - Each image has **one million dimensions**.
  - Even reducing to **1000 hidden units** requires:
    - A fully connected layer with  $10^6 \times 10^3 = 10^9 \text{ parameters}$
  - Such a model would require:
    - **Many GPUs**.
    - Expertise in **distributed optimization**.
    - A **lot of time** and **resources**.
  - This makes training **infeasible** in many practical settings.

- One might argue that **lower image resolution** could help:
  - Using **100,000 pixels** instead of 1 million may reduce input size.
  - However, having only **1000 hidden units** likely **underestimates** the model capacity required to:
    - Learn **good representations** of images.
    - Build **effective classifiers**.

- A practical image classification system will likely still need **billions of parameters**:
  - This would require **massive datasets** to avoid overfitting.
  - Yet, in practice, **humans and machines** can distinguish cats from dogs **quite easily**.

- This apparent contradiction is resolved by the fact that:
  - **Images exhibit rich structure**.
  - That structure can be **exploited** by:
    - **Human perception**.
    - **Machine learning models**.

- **Convolutional Neural Networks (CNNs)** are a **creative approach** for exploiting the **known structure** in natural images.


## **Invariance**

- When detecting an object in an image:
  - The method should not depend too heavily on the **precise location** of the object.
  - Ideally, the system should **exploit location invariance**.

- **Examples**:
  - Pigs usually don’t fly, and planes don’t swim.
  - But we should still **recognize a pig** even if it appears at the **top of the image**.

- **Analogy: "Where’s Waldo?"**:
  - A children's game involving **finding Waldo** in **busy scenes**.
  - Waldo always wears a **distinctive outfit**.
  - He appears in **unexpected locations**.
  - The challenge lies not in what he looks like, but in **where he appears**.

- **Invariance insight**:
  - Waldo’s **appearance is independent of his location**.
  - We could scan the image using a **Waldo detector**:
    - Assigns a **score to each patch** indicating likelihood of Waldo’s presence.
  - Many **object detection** and **segmentation algorithms** use this strategy :cite:`Long.Shelhamer.Darrell.2015`.

- **CNNs and spatial invariance**:
  - CNNs **systematize spatial invariance**.
  - They exploit this property to:
    - Learn **useful representations**.
    - Use **fewer parameters**.

![Can you find Waldo (image courtesy of William Murphy (Infomatique))?](../img/waldo-football.jpg)
:width:`400px`
:label:`img_waldo`


- We can now make these intuitions more concrete by listing key **design principles** for a neural network architecture suited to **computer vision**:

  1. **Translation invariance (or translation equivariance)**:
 
     - The network's earliest layers should **respond similarly to the same patch**, regardless of **where it appears** in the image.

  3. **Locality principle**:
     - The earliest layers should focus on **local regions** of the image.
     - These layers should **ignore distant regions** at first.
     - Later layers can **aggregate local representations** to make **whole-image predictions**.

  3. **Hierarchical feature representation**:
     - Deeper layers should capture **longer-range features**.
     - This mimics **higher-level visual processing** in biological systems.

- Let's now see how these ideas **translate into mathematics**.


## **Constraining the MLP**

- We begin with an **MLP** that takes **two-dimensional images** $\mathbf{X}$ as input:
  - The immediate hidden representations $\mathbf{H}$ are also represented as **matrices** (2D tensors in code).
  - Both $\mathbf{X}$ and $\mathbf{H}$ are assumed to have the **same shape**.
  - This implies that **hidden representations also possess spatial structure**.

- Let $[\mathbf{X}]_{i, j}$ and $[\mathbf{H}]_{i, j}$ denote the pixel at location $(i,j)$ in:
  - The input image.
  - The hidden representation.

- To allow **each hidden unit to receive input from every pixel**:
  - We switch from using **weight matrices** (as in MLPs) to **fourth-order weight tensors** $\mathsf{W}$.
  - Let $\mathbf{U}$ contain the **biases**.

- The fully connected layer can be written as:

  $$
  \begin{aligned}
  \left[\mathbf{H}\right]_{i, j} &= [\mathbf{U}]_{i, j} + \sum_k \sum_l[\mathsf{W}]_{i, j, k, l}  [\mathbf{X}]_{k, l} \\
  &=  [\mathbf{U}]_{i, j} + \sum_a \sum_b [\mathsf{V}]_{i, j, a, b}  [\mathbf{X}]_{i+a, j+b}.
  \end{aligned}
  $$

- The switch from $\mathsf{W}$ to $\mathsf{V}$ is **cosmetic**:
  - There is a **one-to-one correspondence** between the coefficients of the two fourth-order tensors.
  - We re-index the subscripts $(k, l)$ such that:
    - $k = i + a$
    - $l = j + b$
    - So $[\mathsf{V}]_{i, j, a, b} = [\mathsf{W}]_{i, j, i+a, j+b}$

- The indices $a$ and $b$:
  - Run over both **positive and negative offsets**.
  - Cover the **entire image**.

- For any given location $(i, j)$ in the hidden representation $[\mathbf{H}]_{i, j}$:
  - Its value is computed by:
    - Summing over nearby pixels in $\mathbf{X}$.
    - Weighted by the corresponding elements in $[\mathsf{V}]_{i, j, a, b}$.

- **Parameter count concern**:
  - A single layer mapping a $1000 \times 1000$ image to a $1000 \times 1000$ hidden representation:
    - Requires $10^{12}$ parameters.
    - This is **far beyond current computational capabilities**.


### **Translation Invariance**

- Let's now apply the **first principle**: **translation invariance** :cite:`Zhang.ea.1988`.

- Translation invariance means:
  - A **shift in the input** $\mathbf{X}$ should produce a **shift in the hidden representation** $\mathbf{H}$.
  - For this to hold, the parameters must be **independent of spatial location $(i, j)$**.

- Therefore, we set:
  - $[\mathsf{V}]_{i, j, a, b} = [\mathbf{V}]_{a, b}$
  - $\mathbf{U}$ becomes a **constant**, say $u$

- Under this simplification, the definition of $\mathbf{H}$ becomes:

  $$
  [\mathbf{H}]_{i, j} = u + \sum_a\sum_b [\mathbf{V}]_{a, b}  [\mathbf{X}]_{i+a, j+b}.
  $$

- This is a **convolution**:
  - We are **weighting nearby pixels** at $(i+a, j+b)$
  - Using coefficients $[\mathbf{V}]_{a, b}$
  - To obtain the value at location $[\mathbf{H}]_{i, j}$

- **Parameter reduction**:
  - $[\mathbf{V}]_{a, b}$ has **far fewer coefficients** than $[\mathsf{V}]_{i, j, a, b}$:
    - It no longer depends on $(i, j)$
  - Instead of $10^{12}$ parameters, we now need only about $4 \times 10^6$:
    - Since $a, b \in (-1000, 1000)$

- **Conclusion**:
  - This represents **significant progress** in reducing model complexity.
  - **Time-delay neural networks (TDNNs)** were among the first to exploit this idea :cite:`Waibel.Hanazawa.Hinton.ea.1989`.


### **Locality**

- Now let's apply the second principle: **locality**.

- **Motivation**:
  - To determine what is happening at $[\mathbf{H}]_{i, j}$, we **should not need to look far** from location $(i, j)$.
  - **Distant pixels** are likely to be **less relevant**.

- **Mathematical expression of locality**:
  - For locations where $|a| > \Delta$ or $|b| > \Delta$, we set:
    - $[\mathbf{V}]_{a, b} = 0$
  - This constrains the convolution to a **local window**.

- **Rewritten expression for $[\mathbf{H}]_{i, j}$**:

  $$
  [\mathbf{H}]_{i, j} = u + \sum_{a = -\Delta}^{\Delta} \sum_{b = -\Delta}^{\Delta} [\mathbf{V}]_{a, b}  [\mathbf{X}]_{i+a, j+b}.
  $$
  :eqlabel:`eq_conv-layer`


- This change **reduces the number of parameters** from $4 \times 10^6$ to $4 \Delta^2$:
  - Where $\Delta$ is typically **less than 10**.
  - We achieve a reduction by **four orders of magnitude**.

- Note that :eqref:`eq_conv-layer` defines what is known as a **convolutional layer**.

- **Convolutional Neural Networks (CNNs)**:
  - A **special family of neural networks** that contain **convolutional layers**.
  - In deep learning terminology:
    - $\mathbf{V}$ is referred to as a **convolution kernel**, a **filter**, or simply the layer’s **weights**.
    - These weights are **learnable parameters**.

- **Advantages of convolutional layers**:
  - Previously, representing a single image-processing layer might have required **billions of parameters**.
  - Now, we typically need only a **few hundred**, **without changing the dimensionality** of:
    - The **input**.
    - The **hidden representation**.

- **Tradeoff**:
  - We gain parameter efficiency, but:
    - Our features become **translation invariant**.
    - Each layer can incorporate only **local information** for each hidden unit.

- **On inductive bias**:
  - All learning depends on imposing **inductive bias**.
  - If the bias **matches reality** (e.g., translation invariance in images), the model is:
    - **Sample-efficient**.
    - Capable of **generalizing well** to unseen data.
  - If the bias is **misaligned with reality**, the model may:
    - **Struggle to fit training data**.
    - **Generalize poorly**.

- This **dramatic reduction in parameters** leads us to the final principle:
  - **Deeper layers should capture larger and more complex aspects** of an image.
  - This is achieved by:
    - **Interleaving nonlinearities** with **convolutional layers** repeatedly.


## **Convolutions**

- Let's briefly review **why** :eqref:`eq_conv-layer` is called a **convolution**.

- In mathematics, the **convolution between two functions** $f, g: \mathbb{R}^d \to \mathbb{R}$ is defined as :cite:`Rudin.1973`:

  $$
  (f * g)(\mathbf{x}) = \int f(\mathbf{z}) g(\mathbf{x}-\mathbf{z}) d\mathbf{z}.
  $$

- **Interpretation**:
  - This measures the **overlap** between $f$ and $g$
  - When one function is **flipped** and **shifted** by $\mathbf{x}$.

- For **discrete functions**, the integral becomes a **sum**:
  - For infinite-dimensional, square-summable vectors indexed by $\mathbb{Z}$:

    $$
    (f * g)(i) = \sum_a f(a) g(i-a).
    $$

- For **two-dimensional tensors**, the convolution becomes:

  $$
  (f * g)(i, j) = \sum_a\sum_b f(a, b) g(i-a, j-b).
  $$
  :eqlabel:`eq_2d-conv-discrete`

- **Comparison to :eqref:`eq_conv-layer`**:
  - :eqref:`eq_conv-layer` uses $(i+a, j+b)$ rather than $(i-a, j-b)$.
  - This difference is **mostly cosmetic**.
  - The two forms can be matched through **notation adjustment**.

- **Technical note**:
  - The operation in :eqref:`eq_conv-layer` is more accurately a **cross-correlation**.
  - We will return to this distinction in the **next section**.


## Channels
:label:`subsec_why-conv-channels`

- Returning to our **Waldo detector**, let's see what the process looks like.

- A **convolutional layer**:
  - Picks **windows of a given size** from the image.
  - Weighs **pixel intensities** according to the **filter** $\mathsf{V}$.

- As illustrated in :numref:`fig_waldo_mask`:
  - The goal is to **learn a filter** such that:
    - Wherever the "**waldoness**" is highest,
    - There is a **peak in the hidden layer representations**.

![Detect Waldo (image courtesy of William Murphy (Infomatique)).](../img/waldo-mask.jpg)
:width:`400px`
:label:`fig_waldo_mask`


- There is one issue with the approach so far:
  - We have **ignored the fact that images have three channels**: red, green, and blue.

- As a result:
  - Images are not just 2D arrays, but **third-order tensors**.
  - They have **height**, **width**, and **channel** dimensions.
  - For example, an image might have shape $1024 \times 1024 \times 3$.

- In this structure:
  - The first two axes capture **spatial relationships**.
  - The third axis provides a **multidimensional representation** at each pixel.

- We index the image tensor as $[\mathsf{X}]_{i, j, k}$.

- The **convolutional filter** must adapt:
  - Instead of $[\mathbf{V}]_{a,b}$, we now use $[\mathsf{V}]_{a,b,c}$.


- Since the **input** is a third-order tensor, it's also useful to represent the **hidden representations** as third-order tensors $\mathsf{H}$.

- This means:
  - Instead of assigning a **single hidden value** to each spatial location,
  - We assign a **vector of hidden values** to each location.

- The hidden representation can be viewed as:
  - A stack of **2D grids**, one for each channel.

- These are referred to as:
  - **Channels**, similar to the input.
  - Or **feature maps**, since each channel provides a **spatial map of learned features**.

- **Intuition**:
  - In lower layers near the input:
    - Some channels may specialize in detecting **edges**.
    - Others may focus on identifying **textures**.


- To support **multiple channels** in both inputs ($\mathsf{X}$) and hidden representations ($\mathsf{H}$):
  - We introduce a **fourth coordinate** in the filter tensor $\mathsf{V}$:
    - $[\mathsf{V}]_{a, b, c, d}$

- Putting everything together:

  $$
  [\mathsf{H}]_{i,j,d} = \sum_{a = -\Delta}^{\Delta} \sum_{b = -\Delta}^{\Delta} \sum_c [\mathsf{V}]_{a, b, c, d} [\mathsf{X}]_{i+a, j+b, c},
  $$
  :eqlabel:`eq_conv-layer-channels`

- In this equation:
  - $d$ indexes the **output channels** of the hidden representation $\mathsf{H}$.
  - The next convolutional layer will take $\mathsf{H}$ (a **third-order tensor**) as input.

- We adopt :eqref:`eq_conv-layer-channels` as the **definition of a convolutional layer with multiple channels**:
  - $\mathsf{V}$ serves as the **kernel or filter** of the layer.

- There are still many operations to address:
  - How to **combine hidden representations** into a single output (e.g., detecting whether Waldo is **anywhere** in the image).
  - How to **compute efficiently**.
  - How to **stack multiple layers**.
  - How to choose appropriate **activation functions**.
  - How to make **practical design decisions** for effective networks.

- We explore these topics in the remainder of the chapter.


## Summary and Discussion

- In this section, we **derived the structure of convolutional neural networks from first principles**.
  - While it is unclear if CNNs were originally invented this way,
  - It is satisfying to see they are the **right choice** when following **reasonable principles** for image processing, particularly at lower levels.

- Key principles:
  - **Translation invariance**: all patches in an image are **treated equally**, regardless of position.
  - **Locality**: only a **small neighborhood** of pixels is used to compute each hidden representation.

- **Historical note**:
  - One of the earliest forms of CNNs appeared as the **Neocognitron** :cite:`Fukushima.1982`.

- A second principle we discussed:
  - **Reducing the number of parameters** without compromising **expressive power**, under suitable assumptions.
  - This led to a **dramatic reduction in complexity**, making previously infeasible problems **tractable**.

- **Adding channels**:
  - Helps regain some complexity lost due to the **locality** and **invariance** constraints.
  - It is natural to use **additional channels** beyond red, green, and blue:
    - For example, **hyperspectral satellite images** may contain **tens to hundreds of channels**,
    - Reporting measurements across **many different wavelengths**.

- In the following, we will learn:
  - How to use convolutions to **manipulate image dimensionality**,
  - How to move from **location-based to channel-based representations**,
  - How to handle **large numbers of categories efficiently**.
